In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from google.colab import drive

drive.mount('/content/drive')

MODEL_NAME    = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
MAX_Q_LENGTH  = 128
MAX_C_LENGTH  = 384
BATCH_SIZE    = 16
C             = 10.0
CLASS_WEIGHT  = 'balanced'
CTX_LIMIT     = 3
POOLING       = 'mean'
COMBINE_MODES = ['concat', 'diff', 'prod', 'concat_diff']
DATA_ROOT     = '/content/drive/MyDrive/data'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert_model.eval()

def load_and_preprocess(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    rows = []
    label_mapping = {"yes": 1, "no": 0, "maybe": 2}
    for pmid, item in data.items():
        question = item["QUESTION"]
        contexts = " ".join(item["CONTEXTS"][:CTX_LIMIT])
        rows.append({
            "pmid"    : pmid,
            "question": question,
            "context" : contexts,
            "label"   : label_mapping[item["final_decision"]]
        })
    return pd.DataFrame(rows)

def encode_texts(texts, max_length):
    all_embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch   = texts[i:i+BATCH_SIZE]
        encoded = tokenizer(batch, max_length=max_length, truncation=True,
                            padding='max_length', return_tensors='pt')
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            outputs = bert_model(**encoded)
        if POOLING == 'cls':
            embeddings = outputs.last_hidden_state[:, 0, :]
        else:
            mask       = encoded['attention_mask'].unsqueeze(-1)
            embeddings = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
        all_embeddings.append(embeddings.cpu().numpy())
    return np.vstack(all_embeddings)

def combine_vectors(q_vecs, c_vecs, mode):
    if mode == 'concat':
        return np.concatenate([q_vecs, c_vecs], axis=1)
    elif mode == 'diff':
        return q_vecs - c_vecs
    elif mode == 'prod':
        return q_vecs * c_vecs
    elif mode == 'concat_diff':
        return np.concatenate([q_vecs, c_vecs, q_vecs - c_vecs], axis=1)

all_train_dfs = []
for fold in range(10):
    fold_dir = f'{DATA_ROOT}/pqal_fold{fold}'
    all_train_dfs.append(load_and_preprocess(f'{fold_dir}/train_set.json'))
    all_train_dfs.append(load_and_preprocess(f'{fold_dir}/dev_set.json'))

full_train_df = pd.concat(all_train_dfs, ignore_index=True).drop_duplicates(subset='pmid')
test_df       = load_and_preprocess(f'{DATA_ROOT}/test_set.json')

q_train = encode_texts(full_train_df['question'].tolist(), MAX_Q_LENGTH)
c_train = encode_texts(full_train_df['context'].tolist(),  MAX_C_LENGTH)
q_test  = encode_texts(test_df['question'].tolist(),       MAX_Q_LENGTH)
c_test  = encode_texts(test_df['context'].tolist(),        MAX_C_LENGTH)

y_train = full_train_df['label'].values
y_test  = test_df['label'].values

for mode in COMBINE_MODES:
    X_train = combine_vectors(q_train, c_train, mode)
    X_test  = combine_vectors(q_test,  c_test,  mode)

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    model = LogisticRegression(max_iter=1000, C=C, class_weight=CLASS_WEIGHT, solver='lbfgs')
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc        = accuracy_score(y_test, preds)
    p, r, f, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)

    print(f"\nCombination mode: {mode}")
    print(f"Accuracy: {acc:.3f} | F1 macro: {f:.3f}")
    print(classification_report(y_test, preds, target_names=['no', 'yes', 'maybe'], digits=3, zero_division=0))
    cm = confusion_matrix(y_test, preds)
    print("Confusion matrix:")
    print(cm)
    print("Confusion matrix (normalised):")
    print(np.round(cm.astype(float) / cm.sum(axis=1, keepdims=True), 3))

    pd.DataFrame({
        'pmid'   : test_df['pmid'].values,
        'y_true' : y_test,
        'y_pred' : preds,
        'correct': (preds == y_test).astype(int)
    }).to_csv(f'{DATA_ROOT}/exp1_test_results.csv', index=False)